In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
"""
catev_model_v2_training
=================

"""

import sys
import warnings
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# ── Constants ──────────────────────────────────────────────────────────────────
SLOPE_WINDOWS = {"2m": 60, "5m": 150, "7m": 210, "15m": 450}
LAG_15M = 450

LAG_VITALS = [
    "spo2", "heart_rate", "resp_rate_smoothed",
    "sbp", "dbp", "mbp", "etco2", "pulse_pressure", "combined_score",
]

DROP_ALWAYS = {
    "patient_id", "age", "time",
    "resp_rate",                          # raw RR excluded; smoothed kept
    "severity_sum",
    "z_spo2", "z_hr", "z_rr", "z_sbp", "z_dbp", "z_mbp", "z_etco2", "z_pp",
    "s_spo2", "s_hr", "s_rr", "s_sbp", "s_dbp", "s_mbp", "s_etco2", "s_pp",
    "selected_cond_1", "selected_cond_2",
    "severity_label", "result_label",
    "etco2_prev", "spo2_prev", "heart_rate_prev",
    "future_label",
}


# ── Feature engineering ────────────────────────────────────────────────────────

def add_rolling_combined(df):
    """Rolling mean/std (min_periods=2) and min/max (min_periods=1)
    of combined_score at each window. Patient-wise when patient_id present."""
    grp_col = "patient_id" if "patient_id" in df.columns else None

    for win_name, win_size in SLOPE_WINDOWS.items():
        if grp_col:
            g = df.groupby(grp_col)["combined_score"]
            df[f"roll_mean_{win_name}_combined"] = g.transform(
                lambda x, w=win_size: x.rolling(w, min_periods=2).mean())
            df[f"roll_std_{win_name}_combined"] = g.transform(
                lambda x, w=win_size: x.rolling(w, min_periods=2).std())
        else:
            df[f"roll_mean_{win_name}_combined"] = (
                df["combined_score"].rolling(win_size, min_periods=2).mean())
            df[f"roll_std_{win_name}_combined"] = (
                df["combined_score"].rolling(win_size, min_periods=2).std())

    # min / max only at the 15m window; min_periods=1 so every row is populated
    if grp_col:
        g = df.groupby(grp_col)["combined_score"]
        df["roll_min_15m_combined"] = g.transform(
            lambda x: x.rolling(LAG_15M, min_periods=1).min())
        df["roll_max_15m_combined"] = g.transform(
            lambda x: x.rolling(LAG_15M, min_periods=1).max())
    else:
        df["roll_min_15m_combined"] = (
            df["combined_score"].rolling(LAG_15M, min_periods=1).min())
        df["roll_max_15m_combined"] = (
            df["combined_score"].rolling(LAG_15M, min_periods=1).max())

    return df


def add_lag_vitals(df):
    """Add lag_15m_<vital> = value 450 rows ago per patient.
    Early rows (< 450 predecessors) use that patient's first-row value."""
    grp_col = "patient_id" if "patient_id" in df.columns else None

    def _shift_fill_first(s):
        return s.shift(LAG_15M).fillna(s.iloc[0])

    for col in LAG_VITALS:
        if col not in df.columns:
            continue
        if grp_col:
            df[f"lag_15m_{col}"] = df.groupby(grp_col)[col].transform(
                _shift_fill_first)
        else:
            df[f"lag_15m_{col}"] = _shift_fill_first(df[col])

    return df


def build_feature_matrix(df):
    """Add engineered features, drop excluded columns. Returns (X, y, feat_names)."""
    df = add_rolling_combined(df.copy())
    df = add_lag_vitals(df)

    y = df["future_label"].astype(int)
    drop_cols = DROP_ALWAYS & set(df.columns)
    X = df.drop(columns=list(drop_cols))

    obj_cols = X.select_dtypes(include="object").columns.tolist()
    if obj_cols:
        print(f"  [warn] dropping object columns: {obj_cols}")
        X = X.drop(columns=obj_cols)

    return X, y, X.columns.tolist()


# ── Helpers ────────────────────────────────────────────────────────────────────

def sample_weights_balanced(y):
    """Balanced per-sample weights from the provided labels only."""
    return compute_sample_weight("balanced", y)


# ── Model factory ──────────────────────────────────────────────────────────────

def make_hgb():
    return HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, max_depth=6,
        min_samples_leaf=40, l2_regularization=0.1, max_bins=255,
        early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=20, random_state=42,
    )


# ── Cross-validation (row-wise StratifiedKFold) ────────────────────────────────

def cv_evaluate(X_tr, y_tr, model_fn, n_splits=5, label="Model"):
    """
    Standard row-wise StratifiedKFold CV on train rows.
    Each row is self-contained (history already encoded in features), so
    row-wise splitting is valid and gives an honest performance estimate.
    Class weights recomputed from each fold's train rows only.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_f1s, fold_balacc = [], []

    for fold, (fi_tr, fi_va) in enumerate(skf.split(X_tr, y_tr), 1):
        Xf_tr, Xf_va = X_tr.iloc[fi_tr], X_tr.iloc[fi_va]
        yf_tr, yf_va = y_tr.iloc[fi_tr], y_tr.iloc[fi_va]

        sw = sample_weights_balanced(yf_tr.values)
        clf = model_fn()
        clf.fit(Xf_tr, yf_tr, sample_weight=sw)
        preds = clf.predict(Xf_va)

        f1 = f1_score(yf_va, preds, average="macro", zero_division=0)
        ba = balanced_accuracy_score(yf_va, preds)
        fold_f1s.append(f1)
        fold_balacc.append(ba)

        cls, cts = np.unique(yf_va, return_counts=True)
        dist = " | ".join(
            f"{['N','C','E'][int(c)]}={n}({100*n/len(yf_va):.0f}%)"
            for c, n in zip(cls, cts))
        print(f"  Fold {fold}/{n_splits}  macro-F1={f1:.4f}  bal-acc={ba:.4f}"
              f"  [{dist}]")

    print(f"\n  {label} -- CV macro-F1 : {np.mean(fold_f1s):.4f}"
          f" +/- {np.std(fold_f1s):.4f}")
    print(f"  {label} -- CV bal-acc  : {np.mean(fold_balacc):.4f}"
          f" +/- {np.std(fold_balacc):.4f}")

    return fold_f1s


# ── Main pipeline ──────────────────────────────────────────────────────────────

def run(input_csv, output_prefix="catev_v2"):
    print(f"\n{'='*60}")
    print("  CatevCode v2 -- Severity Classifier")
    print(f"{'='*60}\n")

    df = pd.read_csv(input_csv)
    print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} cols")

    # Label distribution
    print("\nTarget distribution (future_label):")
    vc = df["future_label"].value_counts().sort_index()
    for lbl, cnt in vc.items():
        name = {0: "Normal", 1: "Critical", 2: "Emergency"}.get(lbl, str(lbl))
        print(f"  {lbl} ({name:>9}): {cnt:6d}  ({100*cnt/len(df):5.1f}%)")

    # Build feature matrix
    X, y, feat_names = build_feature_matrix(df)
    print(f"\nFeature matrix : {X.shape[0]} rows x {X.shape[1]} features")
    print(f"  lag-15m cols : {[c for c in feat_names if c.startswith('lag_15m_')]}")
    print(f"  roll min/max : {[c for c in feat_names if 'min' in c or 'max' in c]}")

    # ── Row-wise stratified 80/20 hold-out split ──────────────────────────────
    print("\n[Split] Row-wise stratified 80/20 hold-out:")
    idx = np.arange(len(X))
    tr_idx, te_idx = train_test_split(
        idx, test_size=0.20, random_state=42, stratify=y.values)

    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    for split_name, ys in [("Train", y_tr), ("Test ", y_te)]:
        cls, cts = np.unique(ys, return_counts=True)
        dist = "  ".join(
            f"{['N','C','E'][int(c)]}={n}({100*n/len(ys):.0f}%)"
            for c, n in zip(cls, cts))
        print(f"  {split_name}: {len(ys):7d} rows   [{dist}]")

    # Class weights from train rows only
    classes_tr, counts_tr = np.unique(y_tr, return_counts=True)
    class_w_tr = len(y_tr) / (len(classes_tr) * counts_tr)
    print("\nClass weights (balanced, from train rows only):")
    for c, cnt, w in zip(classes_tr, counts_tr, class_w_tr):
        name = {0:"Normal", 1:"Critical", 2:"Emergency"}.get(c, str(c))
        print(f"  class {c} ({name:>9}, n={cnt:6d}) -> weight = {w:.4f}")

    # ── StratifiedKFold(5) CV on train rows ───────────────────────────────────
    print("\n[CV] StratifiedKFold(5) row-wise on train rows:")

    print(f"\n{'--'*25}")
    print("MODEL: HistGradientBoostingClassifier")
    print(f"{'--'*25}")
    hgb_f1s = cv_evaluate(X_tr, y_tr, make_hgb, label="HGB")

    hgb_cv = float(np.mean(hgb_f1s))

    # ── Train on full train set, eval on hold-out ─────────────────────────────
    print(f"\n>>> HistGradientBoosting (CV macro-F1 {hgb_cv:.4f})")
    print(f"\n[Train] Training HistGradientBoosting on all train rows ...")

    sw_tr = sample_weights_balanced(y_tr.values)
    clf = make_hgb()
    clf.fit(X_tr, y_tr, sample_weight=sw_tr)

    test_preds = clf.predict(X_te)
    test_f1 = f1_score(y_te, test_preds, average="macro", zero_division=0)
    test_ba = balanced_accuracy_score(y_te, test_preds)
    print(f"  Hold-out test macro-F1={test_f1:.4f}  bal-acc={test_ba:.4f}")

    print("\nHold-out test classification report (HistGradientBoosting):")
    report = classification_report(
        y_te, test_preds,
        target_names=["Normal(0)", "Critical(1)", "Emergency(2)"],
        zero_division=0,
    )
    print(report)

    print("Confusion matrix (rows=true, cols=pred):")
    cm = confusion_matrix(y_te, test_preds)
    print(pd.DataFrame(cm,
                       index=["True-N", "True-C", "True-E"],
                       columns=["Pred-N", "Pred-C", "Pred-E"]))

    # ── Retrain on ALL data for deployment ────────────────────────────────────
    print("\n[Final] Retraining HistGradientBoosting on full dataset for deployment ...")
    sw_all = sample_weights_balanced(y.values)
    final_clf = make_hgb()
    final_clf.fit(X, y, sample_weight=sw_all)

    # Feature importances via permutation (HGB has no built-in feature_importances_)
    print("\nTop-40 feature importances (permutation importance on a 20k-row sample):")
    sidx = np.random.default_rng(0).choice(len(X), min(20_000, len(X)), replace=False)
    perm = permutation_importance(
        final_clf, X.iloc[sidx], y.iloc[sidx],
        n_repeats=5, scoring="balanced_accuracy",
        random_state=0, n_jobs=-1,
    )
    imp = pd.Series(perm.importances_mean,
                    index=feat_names).sort_values(ascending=False)

    top40 = imp.head(40)
    for feat, val in top40.items():
        bar = "#" * max(1, int(val / top40.max() * 30))
        print(f"  {feat:<55s} {val:.5f}  {bar}")

    # Save model
    model_path = f"{output_prefix}_model.pkl"
    with open(model_path, "wb") as fh:
        pickle.dump({"model": final_clf, "features": feat_names,
                     "winner": "HistGradientBoosting"}, fh)
    print(f"\nModel saved -> {model_path}")

    # Write report
    report_path = f"{output_prefix}_report.txt"
    with open(report_path, "w") as fh:
        fh.write("CATEV v2 SEVERITY CLASSIFIER -- TRAINING REPORT\n")
        fh.write("=" * 60 + "\n\n")
        fh.write(f"Dataset : {input_csv}\n")
        fh.write(f"Rows    : {len(df)}   Features used: {len(feat_names)}\n\n")

        fh.write("CLASS DISTRIBUTION (full dataset)\n")
        for lbl, cnt in vc.items():
            name = {0:"Normal",1:"Critical",2:"Emergency"}.get(lbl, str(lbl))
            fh.write(f"  {lbl} ({name}): {cnt}\n")

        fh.write("\nSPLIT STRATEGY\n")
        fh.write("  Row-wise stratified 80/20 hold-out (no patient grouping).\n")
        fh.write("  Each row encodes its own temporal context via rolling/lag\n")
        fh.write("  features, so row-wise splitting is valid.\n")
        fh.write(f"  Train rows : {len(y_tr)}    Test rows : {len(y_te)}\n")

        fh.write("\nCLASS WEIGHTS (balanced, from train rows only)\n")
        for c, w in zip(classes_tr, class_w_tr):
            fh.write(f"  class {c}: {w:.4f}\n")

        fh.write(f"\nHGB  CV macro-F1 : {np.mean(hgb_f1s):.4f}"
                 f" +/- {np.std(hgb_f1s):.4f}\n")
        fh.write(f"\nModel: HistGradientBoosting\n")
        fh.write(f"Hold-out test macro-F1 : {test_f1:.4f}"
                 f"   bal-acc : {test_ba:.4f}\n\n")

        fh.write("HOLD-OUT TEST CLASSIFICATION REPORT\n")
        fh.write(report + "\n")

        fh.write("CONFUSION MATRIX (rows=true, cols=pred)\n")
        fh.write(str(pd.DataFrame(
            cm,
            index=["True-Normal", "True-Critical", "True-Emergency"],
            columns=["Pred-Normal", "Pred-Critical", "Pred-Emergency"],
        )))

        fh.write("\n\nTOP-40 FEATURE IMPORTANCES\n")
        for feat, val in top40.items():
            fh.write(f"  {feat:<55s} {val:.6f}\n")

        fh.write("\n\nFEATURES USED (full list)\n")
        for fn in feat_names:
            fh.write(f"  {fn}\n")

    print(f"Report saved -> {report_path}")
    print("\nDone.\n")
    return final_clf, feat_names


# ── Entry point ────────────────────────────────────────────────────────────────

def _is_notebook():
    try:
        shell = get_ipython().__class__.__name__   # type: ignore[name-defined]
        return shell in ("ZMQInteractiveShell", "TerminalInteractiveShell", "Shell")
    except NameError:
        return False


if __name__ == "__main__":
    if _is_notebook():
        input_csv = "/kaggle/input/datasets/arjunmahesh09999/final/FINAL_DATASET.csv"
        prefix    = "catev_v2"
    else:
        args = [a for a in sys.argv[1:] if not a.startswith("-")]
        input_csv = args[0] if args else \
            "/kaggle/input/datasets/arjunmahesh09999/final/FINAL_DATASET.csv"
        prefix    = args[1] if len(args) >= 2 else "catev_v2"

    run(input_csv, prefix)


  CatevCode v2 -- Severity Classifier

Loaded: 211522 rows x 86 cols

Target distribution (future_label):
  0.0 (   Normal):  85993  ( 40.7%)
  1.0 ( Critical):  49831  ( 23.6%)
  2.0 (Emergency):  75698  ( 35.8%)

Feature matrix : 211522 rows x 76 features
  lag-15m cols : ['lag_15m_spo2', 'lag_15m_heart_rate', 'lag_15m_resp_rate_smoothed', 'lag_15m_sbp', 'lag_15m_dbp', 'lag_15m_mbp', 'lag_15m_etco2', 'lag_15m_pulse_pressure', 'lag_15m_combined_score']
  roll min/max : ['roll_min_15m_combined', 'roll_max_15m_combined']

[Split] Row-wise stratified 80/20 hold-out:
  Train:  169217 rows   [N=68794(41%)  C=39865(24%)  E=60558(36%)]
  Test :   42305 rows   [N=17199(41%)  C=9966(24%)  E=15140(36%)]

Class weights (balanced, from train rows only):
  class 0 (   Normal, n= 68794) -> weight = 0.8199
  class 1 ( Critical, n= 39865) -> weight = 1.4149
  class 2 (Emergency, n= 60558) -> weight = 0.9314

[CV] StratifiedKFold(5) row-wise on train rows:

-------------------------------------------